<a href="https://colab.research.google.com/github/nour-houda-melkii/Esprit--PIDS-4DS10-2026-DataVerse/blob/Geopolitical_Agent/2_Data_Cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================================
# NOTEBOOK 2: DATA CLEANING & FEATURE ENGINEERING
# ============================================================================
# Runtime: 5-10 minutes
# Input: geopolitical_historical_2018_2025_RAW.csv
# Output: geopolitical_historical_2018_2025_CLEAN.csv
# ============================================================================

print("="*80)
print("🧹 GEOPOLITICAL FACTOR: DATA CLEANING")
print("="*80 + "\n")

# ----------------------------------------------------------------------------
# CELL 1: Install Dependencies
# ----------------------------------------------------------------------------

!pip install -q pandas numpy ftfy

print("✅ Dependencies installed!")

# ----------------------------------------------------------------------------
# CELL 2: Upload RAW File
# ----------------------------------------------------------------------------

from google.colab import files
import pandas as pd
import numpy as np
import re
import warnings
warnings.filterwarnings('ignore')

print("\n📂 Upload your RAW CSV file:")
print("(The file you downloaded from Notebook 1)")

uploaded = files.upload()

# Get filename
input_file = list(uploaded.keys())[0]
print(f"\n✅ Uploaded: {input_file}")

# ----------------------------------------------------------------------------
# CELL 3: Load Data
# ----------------------------------------------------------------------------

print("\n📂 Loading data...")
df = pd.read_csv(input_file, encoding='utf-8')
print(f"✅ Loaded {len(df):,} rows")

initial_rows = len(df)

# ----------------------------------------------------------------------------
# CELL 4: Fix Encoding
# ----------------------------------------------------------------------------

print("\n🔤 Fixing encoding issues...")

import ftfy

def fix_encoding(text):
    """Fix mojibake."""
    if pd.isna(text):
        return text
    try:
        return ftfy.fix_text(str(text))
    except:
        return text

if 'title' in df.columns:
    df['title'] = df['title'].apply(fix_encoding)
    print("✅ Encoding fixed!")

# ----------------------------------------------------------------------------
# CELL 5: Remove Duplicates
# ----------------------------------------------------------------------------

print("\n🔄 Removing duplicates...")

before = len(df)
df = df.drop_duplicates(subset=['date', 'title'], keep='first')
removed = before - len(df)

print(f"✅ Removed {removed} duplicates ({len(df):,} remaining)")

# ----------------------------------------------------------------------------
# CELL 6: Handle Missing Values
# ----------------------------------------------------------------------------

print("\n❌ Handling missing values...")

# Critical: must have date
if 'date' in df.columns:
    before_drop = len(df)
    df = df[df['date'].notna()]
    dropped = before_drop - len(df)
    print(f"   Dropped {dropped} rows with missing dates")

# Fill missing values
df['title'] = df['title'].fillna('Event (no title)')
df['currency_primary'] = df['currency_primary'].fillna('USD')
df['time'] = df['time'].fillna('')
df['url'] = df['url'].fillna('URL_NOT_AVAILABLE')
df['source'] = df['source'].fillna('UNKNOWN')

print("✅ Missing values handled!")

# ----------------------------------------------------------------------------
# CELL 7: Classify Events
# ----------------------------------------------------------------------------

print("\n🏷️  Classifying events...")

EVENT_PATTERNS = {
    'war': ['war', 'invasion', 'attack', 'military', 'combat', 'battle', 'strike'],
    'sanctions': ['sanction', 'embargo', 'tariff', 'ban'],
    'election': ['election', 'vote', 'ballot', 'referendum'],
    'trade_deal': ['trade deal', 'agreement', 'treaty', 'pact'],
    'crisis': ['crisis', 'emergency', 'panic', 'crash', 'collapse'],
    'protest': ['protest', 'demonstration', 'riot', 'unrest'],
    'disaster': ['disaster', 'earthquake', 'flood', 'hurricane', 'pandemic'],
    'coup': ['coup', 'overthrow']
}

def classify_event(title):
    if pd.isna(title):
        return 'other'

    title_lower = str(title).lower()

    for event_type, keywords in EVENT_PATTERNS.items():
        for keyword in keywords:
            if keyword in title_lower:
                return event_type

    return 'other'

df['event_type'] = df['title'].apply(classify_event)

print("✅ Events classified!")
print(df['event_type'].value_counts().head())

# ----------------------------------------------------------------------------
# CELL 8: Calculate Severity
# ----------------------------------------------------------------------------

print("\n⚠️  Calculating severity scores...")

def calculate_severity(row):
    title = str(row.get('title', '')).lower()
    event_type = row.get('event_type', 'other')

    score = 5  # baseline

    # Event type
    if event_type == 'war':
        score += 3
    elif event_type in ['crisis', 'sanctions', 'disaster', 'coup']:
        score += 2

    # Keywords
    if 'nuclear' in title:
        score += 5
    if 'invasion' in title:
        score += 3
    if 'terrorism' in title or 'terrorist' in title:
        score += 3
    if 'pandemic' in title:
        score += 2
    if 'collapse' in title or 'crash' in title:
        score += 2

    return min(max(score, 0), 10)

df['severity'] = df.apply(calculate_severity, axis=1)

print(f"✅ Severity calculated! Mean: {df['severity'].mean():.2f}")

# ----------------------------------------------------------------------------
# CELL 9: Detect Safe-Haven Events
# ----------------------------------------------------------------------------

print("\n🛡️  Detecting safe-haven events...")

safe_haven_keywords = [
    'war', 'nuclear', 'terrorism', 'pandemic', 'crisis',
    'crash', 'invasion', 'attack', 'conflict', 'disaster'
]

def is_safe_haven(title):
    if pd.isna(title):
        return False
    title_lower = str(title).lower()
    return any(kw in title_lower for kw in safe_haven_keywords)

df['is_safe_haven'] = df['title'].apply(is_safe_haven)

safe_haven_count = df['is_safe_haven'].sum()
print(f"✅ Safe-haven events: {safe_haven_count:,} ({safe_haven_count/len(df)*100:.1f}%)")

# ----------------------------------------------------------------------------
# CELL 10: Engineer Temporal Features
# ----------------------------------------------------------------------------

print("\n📅 Engineering temporal features...")

df['date_dt'] = pd.to_datetime(df['date'], format='%Y%m%d', errors='coerce')
df['year'] = df['date_dt'].dt.year
df['month'] = df['date_dt'].dt.month
df['day_of_week'] = df['date_dt'].dt.dayofweek
df['quarter'] = df['date_dt'].dt.quarter
df['is_weekend'] = df['day_of_week'].isin([5, 6]).astype(int)

def get_market_session(time_str):
    if pd.isna(time_str) or time_str == '':
        return 'unknown'
    try:
        hour = int(str(time_str)[:2])
        if 0 <= hour < 6:
            return 'asia'
        elif 6 <= hour < 14:
            return 'europe'
        else:
            return 'america'
    except:
        return 'unknown'

df['market_session'] = df['time'].apply(get_market_session)

print("✅ Temporal features added!")

# ----------------------------------------------------------------------------
# CELL 11: Generate FX Signals
# ----------------------------------------------------------------------------

print("\n📈 Generating FX direction signals...")

def predict_fx_direction(row):
    currency = row['currency_primary']
    event_type = row['event_type']
    severity = row['severity']
    is_safe_haven = row['is_safe_haven']

    # Safe-haven logic
    if is_safe_haven:
        if currency in ['JPY', 'CHF', 'USD']:
            return 'UP'
        else:
            return 'DOWN'

    # Event-specific logic
    if event_type in ['war', 'crisis']:
        if currency in ['JPY', 'CHF', 'USD']:
            return 'UP'
        else:
            return 'DOWN'
    elif event_type == 'sanctions':
        return 'DOWN'
    elif event_type == 'trade_deal':
        return 'UP'
    elif event_type == 'election':
        return 'NEUTRAL'
    else:
        if severity >= 7:
            return 'DOWN'
        else:
            return 'NEUTRAL'

df['fx_direction'] = df.apply(predict_fx_direction, axis=1)

print("✅ FX directions generated!")
print(df['fx_direction'].value_counts())

# ----------------------------------------------------------------------------
# CELL 12: Expand to Currency Pairs
# ----------------------------------------------------------------------------

print("\n💱 Expanding to currency pairs...")

ALL_PAIRS = {
    'USD': ['EUR/USD', 'GBP/USD', 'USD/JPY', 'USD/CHF'],
    'EUR': ['EUR/USD', 'EUR/GBP', 'EUR/JPY', 'EUR/CHF'],
    'JPY': ['USD/JPY', 'EUR/JPY', 'GBP/JPY', 'CHF/JPY'],
    'GBP': ['GBP/USD', 'EUR/GBP', 'GBP/JPY', 'GBP/CHF'],
    'CHF': ['USD/CHF', 'EUR/CHF', 'GBP/CHF', 'CHF/JPY']
}

rows = []

for _, row in df.iterrows():
    currency = row['currency_primary']
    fx_direction = row['fx_direction']

    if currency not in ALL_PAIRS:
        continue

    pairs = ALL_PAIRS[currency]

    for pair in pairs:
        base_curr = pair[:3]
        quote_curr = pair[4:]

        # Determine signal
        if currency == base_curr:
            if fx_direction == 'UP':
                signal = f"BUY {pair}"
            elif fx_direction == 'DOWN':
                signal = f"SELL {pair}"
            else:
                signal = f"HOLD {pair}"
        else:
            if fx_direction == 'UP':
                signal = f"SELL {pair}"
            elif fx_direction == 'DOWN':
                signal = f"BUY {pair}"
            else:
                signal = f"HOLD {pair}"

        new_row = row.copy()
        new_row['pair'] = pair
        new_row['trading_signal'] = signal
        rows.append(new_row)

df = pd.DataFrame(rows)

print(f"✅ Expanded to {len(df):,} pair-level rows!")

# ----------------------------------------------------------------------------
# CELL 13: Save Clean Data
# ----------------------------------------------------------------------------

output_file = 'geopolitical_historical_2018_2025_CLEAN.csv'
df.to_csv(output_file, index=False, encoding='utf-8-sig')

print("\n" + "="*80)
print("✅ CLEANING COMPLETE!")
print("="*80)
print(f"Input rows:    {initial_rows:,}")
print(f"Output rows:   {len(df):,}")
print(f"Retention:     {len(df)/initial_rows*100:.1f}%")
print(f"Features:      {len(df.columns)}")
print(f"Output file:   {output_file}")
print("="*80 + "\n")

# Download
from google.colab import files
files.download(output_file)

print("✅ File downloaded!")
print("\n📋 Sample of clean data:")
print(df.head(10))

print("\n✅ NOTEBOOK 2 COMPLETE!")
print("➡️  Next: Run Notebook 3 (Model Training)")

🧹 GEOPOLITICAL FACTOR: DATA CLEANING

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 1.7 MB/s eta 0:00:00
✅ Dependencies installed!

📂 Upload your RAW CSV file:
(The file you downloaded from Notebook 1)


Saving geopolitical_historical_2018_2025_RAW.csv to geopolitical_historical_2018_2025_RAW (1).csv

✅ Uploaded: geopolitical_historical_2018_2025_RAW (1).csv

📂 Loading data...
✅ Loaded 44,750 rows

🔤 Fixing encoding issues...
✅ Encoding fixed!

🔄 Removing duplicates...
✅ Removed 13239 duplicates (31,511 remaining)

❌ Handling missing values...
   Dropped 0 rows with missing dates
✅ Missing values handled!

🏷️  Classifying events...
✅ Events classified!
event_type
other        24164
war           3570
sanctions     2288
crisis         429
election       315
Name: count, dtype: int64

⚠️  Calculating severity scores...
✅ Severity calculated! Mean: 5.72

🛡️  Detecting safe-haven events...
✅ Safe-haven events: 4,779 (15.2%)

📅 Engineering temporal features...
✅ Temporal features added!

📈 Generating FX direction signals...
✅ FX directions generated!
fx_direction
NEUTRAL    23592
DOWN        5038
UP          2881
Name: count, dtype: int64

💱 Expanding to currency pairs...
✅ Expanded to 126,

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ File downloaded!

📋 Sample of clean data:
       date      time currency_primary  \
0  20180109  T094500Z              CHF   
0  20180109  T094500Z              CHF   
0  20180109  T094500Z              CHF   
0  20180109  T094500Z              CHF   
1  20180109  T081500Z              CHF   
1  20180109  T081500Z              CHF   
1  20180109  T081500Z              CHF   
1  20180109  T081500Z              CHF   
2  20180108  T133000Z              CHF   
2  20180108  T133000Z              CHF   

                                               title  \
0  $55 Billion Profit for Giant Asset Manager the...   
0  $55 Billion Profit for Giant Asset Manager the...   
0  $55 Billion Profit for Giant Asset Manager the...   
0  $55 Billion Profit for Giant Asset Manager the...   
1  Swiss National Bank Says It Made CHF54 Billion...   
1  Swiss National Bank Says It Made CHF54 Billion...   
1  Swiss National Bank Says It Made CHF54 Billion...   
1  Swiss National Bank Says It Made CHF54 Bil